In [10]:
import pandas as pd 
import os

In [11]:
#  Set working directory 
os.chdir(r'D:\Mission_Blitzkreig')

In [12]:
# Load your campaign data 
df = pd.read_csv(r'Month_1\12.Load_clean_calculate_metrics\ooh_campaigns.csv')

In [14]:
# Calculate Metrics 
df['ctr'] = (df['clicks'] / df['impressions'] * 100).round(2)
df['roi'] = ((df['revenue'] - df['spend']) /df['spend'] * 100).round(2)
df['profit'] = df['revenue'] - df['spend']

print ("✅ Data Loaded! ")

✅ Data Loaded! 


In [17]:
# Pivot table 1 : Revenue by city & format 
pivot_revenue = pd.pivot_table(
    df,
    values='revenue',
    index='city',
    columns='format',
    aggfunc='sum',
    fill_value=0   
)
print("\n Revenue by city and format:")
print(pivot_revenue)


 Revenue by city and format:
format     4 Sheet  48 Sheet  6 Sheet  Bus Shelter  Digital  Supersite
city                                                                  
Belfast          0         0    45000            0        0      33000
Cork             0         0        0            0    28800          0
Dublin        5600     18900        0            0        0      48000
Galway           0         0        0         2800        0          0
Limerick         0     13500        0            0        0          0
Waterford        0         0     4100            0        0          0


In [22]:
# Pivot table 2 : Average Roi by city and format 
pivot_roi = pd.pivot_table(
    df,
    values='roi',
    index='city', 
    columns='format',
    aggfunc='mean',
    fill_value=0
).round(1)

print("\n Average ROI by city & format")
print(pivot_roi)


 Average ROI by city & format
format     4 Sheet  48 Sheet  6 Sheet  Bus Shelter  Digital  Supersite
city                                                                  
Belfast        0.0       0.0   2900.0          0.0      0.0      432.3
Cork           0.0       0.0      0.0          0.0    397.6        0.0
Dublin       211.1     350.0      0.0          0.0      0.0      464.7
Galway         0.0       0.0      0.0        194.7      0.0        0.0
Limerick       0.0     335.5      0.0          0.0      0.0        0.0
Waterford      0.0       0.0    241.7          0.0      0.0        0.0


In [23]:
# Export to excel 
with pd.ExcelWriter('advanced_pivot_analysis.xlsx') as writer :
    pivot_revenue.to_excel(writer, sheet_name='Revenue')
    pivot_roi.to_excel(writer, sheet_name='ROI')

    print("Pivot Table Saved !")

Pivot Table Saved !


In [25]:
# Part 2 : Export to excel 
# Create the city metadata 
city_data = pd.DataFrame({
    'city' : ['Dublin', 'Cork', 'Belfast', 'Galway', 'Limerick', 'Waterford'],
    'population' : [1400000, 210000, 345000, 80000, 95000, 54000],
    'region' : ['East', 'South', 'North', 'West', 'Mid', 'South'],
    'avg_income' : [45000, 38000, 35000, 36000, 34000, 33000]
})

# Merge campaign data with city data 
df_merged = df.merge(city_data, on='city', how='left')

print("\n Merged Data :")
print(df_merged[['campaign_name','city','population','region','roi']].head(10))

# Analysis : Does population co-relate with performance  ?
city_analysis = df_merged.groupby('city').agg({
    'revenue': 'sum',
    'roi': 'mean',
    'population': 'first'
}).round(2)

city_analysis['revenue_per_capita'] = (city_analysis['revenue'] / city_analysis['population']).round(2)

print("\n📊 City Analysis with Population:")
print(city_analysis.sort_values('revenue_per_capita', ascending=False))


 Merged Data :
                     campaign_name       city  population region      roi
0    Dublin City Centre - 48 Sheet     Dublin     1400000   East   350.00
1          Cork Motorway - Digital       Cork      210000  South   377.61
2       Belfast Roadside - 6 Sheet    Belfast      345000  North  2900.00
3       Dublin Airport - Supersite     Dublin     1400000   East   464.71
4        Galway City - Bus Shelter     Galway       80000   West   194.74
5       Limerick Retail - 48 Sheet   Limerick       95000    Mid   335.48
6            Dublin DART - 4 Sheet     Dublin     1400000   East   211.11
7              Cork City - Digital       Cork      210000  South   417.65
8    Belfast City Hall - Supersite    Belfast      345000  North   432.26
9  Waterford Retail Park - 6 Sheet  Waterford       54000  South   241.67

📊 City Analysis with Population:
           revenue      roi  population  revenue_per_capita
city                                                       
Belfast      780